# NLP Lab Assignment

`N` is detected from the user's own input and every table is sized to exactly that `N`.

**Corpus:** *Pride and Prejudice* by Jane Austen (~121,000 words), plain text from
Project Gutenberg. Save it as `corpus.txt`.download link
`https://raw.githubusercontent.com/GITenberg/Pride-and-Prejudice_1342/master/1342.txt`.

**Covered in this notebook:**

- Text preprocessing: regex cleaning, tokenization
- Stop word removal, Porter stemming, WordNet lemmatization
- Text representation: Bag of Words and TF-IDF
- Edit distance (Levenshtein) and corpus-based spelling correction — with sample tests
- **Dictionary-based** spelling correction beyond the corpus vocabulary — with sample tests
- **Automatic N-gram order detection** from the number of words the user types, with
  *no* fixed upper bound 
- Sentence boundary markers (`<s>`, `</s>`), padded to exactly the detected `N`
- Dynamic N-gram count tables (any order, no hardcoding) — with a sample test
- **Explicit unigram / bigram / trigram / ... MLE probability tables**
- **Laplace (Add-One) smoothing**, generalized for any N, updating those same tables
- Backoff for unseen histories
- **Non-deterministic**, weighted-probability next-word prediction (Shannon's-game style)
- Full autoregressive sentence generation


In [28]:
import re
import math
import random
import sys
import nltk
import numpy as np
import pandas as pd

from collections import defaultdict, Counter
from IPython.display import display
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

for resource, package in [("tokenizers/punkt", "punkt"),
                          ("tokenizers/punkt_tab", "punkt_tab"),
                          ("corpora/stopwords", "stopwords"),
                          ("corpora/wordnet", "wordnet"),
                          ("corpora/omw-1.4", "omw-1.4")]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Corpus

`corpus.txt` contain just the novel's text.

In [17]:
with open("corpus.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

print(f"Corpus length: {len(raw_text):,} characters, "
      f"{len(raw_text.split()):,} whitespace-separated words")
print()
print(raw_text[:500])

Corpus length: 684,805 characters, 121,571 whitespace-separated words

Produced by Anonymous Volunteers





PRIDE AND PREJUDICE

By Jane Austen



Chapter 1


It is a truth universally acknowledged, that a single man in possession
of a good fortune, must be in want of a wife.

However little known the feelings or views of such a man may be on his
first entering a neighbourhood, this truth is so well fixed in the minds
of the surrounding families, that he is considered the rightful property
of some one or other of their daughters.

"My dear Mr. Bennet," said his la


## 2. Base Text Cleaning & Vocabulary

This base vocabulary (no `<s>`/`</s>` markers) is used for spelling correction, the
stopword/stemming/lemmatization demo, and BoW/TF-IDF. It does **not** depend on `N`,
so it can be built before we know how many words will type.Remove bracketed text,-- and ...,digits,characters other than letter spaces and apostrophes,extra spaces.


In [3]:
def clean_sentence(sentence):
    """Regex clean a single sentence: strip bracketed asides, dashes/ellipses,
    digits, and any character that is not a letter or apostrophe."""
    sentence = re.sub(r'\[[^]]*\]', '', sentence)
    sentence = re.sub(r'--|\.\.\.', '', sentence)
    sentence = re.sub(r'\d+', '', sentence)
    sentence = re.sub(r"[^a-zA-Z'\s]", "", sentence)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence.lower()


raw_sentences = sent_tokenize(raw_text)

base_sentences = [clean_sentence(sentence).split() for sentence in raw_sentences]
base_sentences = [sentence for sentence in base_sentences if sentence]  # drop empties

base_tokens = [word for sentence in base_sentences for word in sentence]
base_vocabulary = set(base_tokens)

print("Total sentences:", len(base_sentences))
print("Total base tokens:", len(base_tokens))
print("Base vocabulary size:", len(base_vocabulary))

Total sentences: 5976
Total base tokens: 121477
Base vocabulary size: 6818


## 3. Stop Word Removal, Stemming & Lemmatization

These normalization steps (from Lab 1) are demonstrated here on a raw corpus sentence.

Note: the n-gram generator itself (Section 8 onward) deliberately keeps stop words
and does **not** stem/lemmatize its training tokens — a language model needs the exact
surface word forms and function words (`the`, `is`, `to`, ...) to predict fluent,
grammatical continuations. Stemming/lemmatizing would collapse different word forms
together and break sentence-generation quality. The tools are still fully implemented
and demonstrated below, as required.


In [4]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def full_preprocess(text, remove_stopwords=True, use_lemmatization=True):
    """Lab 1 style preprocessing pipeline: lowercase, regex clean, tokenize,
    optional stop word removal, optional lemmatization."""
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    words = word_tokenize(text)

    if remove_stopwords:
        words = [w for w in words if w not in stop_words]

    if use_lemmatization:
        words = [lemmatizer.lemmatize(w) for w in words]

    return words


# Pick a reasonably long, plain sentence from the corpus for a clear demo
sample_sentence = next(s for s in raw_sentences if 60 < len(s) < 140)
print("Original sentence:", sample_sentence)

tokens_before = word_tokenize(re.sub(r'[^a-zA-Z\s]', '', sample_sentence.lower()))
print("\nTokens (before stop word removal):", tokens_before)

tokens_after_stopwords = [w for w in tokens_before if w not in stop_words]
print("Tokens (after stop word removal): ", tokens_after_stopwords)

stemmed_words = [stemmer.stem(w) for w in tokens_after_stopwords]
lemmatized_words = [lemmatizer.lemmatize(w) for w in tokens_after_stopwords]

print("\nStemming vs Lemmatization")
for original, stem, lemma in zip(tokens_after_stopwords, stemmed_words, lemmatized_words):
    print(f"  {original:<15} stem: {stem:<15} lemma: {lemma}")

Original sentence: "My dear Mr. Bennet," said his lady to him one day, "have you heard that
Netherfield Park is let at last?"

Tokens (before stop word removal): ['my', 'dear', 'mr', 'bennet', 'said', 'his', 'lady', 'to', 'him', 'one', 'day', 'have', 'you', 'heard', 'that', 'netherfield', 'park', 'is', 'let', 'at', 'last']
Tokens (after stop word removal):  ['dear', 'mr', 'bennet', 'said', 'lady', 'one', 'day', 'heard', 'netherfield', 'park', 'let', 'last']

Stemming vs Lemmatization
  dear            stem: dear            lemma: dear
  mr              stem: mr              lemma: mr
  bennet          stem: bennet          lemma: bennet
  said            stem: said            lemma: said
  lady            stem: ladi            lemma: lady
  one             stem: one             lemma: one
  day             stem: day             lemma: day
  heard           stem: heard           lemma: heard
  netherfield     stem: netherfield     lemma: netherfield
  park            stem: park         

## 4. Text Representation: Bag of Words & TF-IDF

Bag of Words (BoW) represents each document as raw word-frequency counts, discarding
grammar and order. TF-IDF additionally down-weights words that are common across many
documents:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D), \qquad
\text{TF}(t, d) = \frac{C(t, d)}{\sum_{t' \in d} C(t', d)}, \qquad
\text{IDF}(t, D) = \log\!\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

Six corpus sentences (treated as separate "documents") are used to demonstrate both
representations below. Matrices are printed with a fixed 3-decimal format, so an
exact zero shows as `0.000` rather than a bare `0.`.


In [5]:
demo_documents = [
    " ".join(full_preprocess(sentence, remove_stopwords=True, use_lemmatization=True))
    for sentence in raw_sentences[:6]
]

print("--- 1. Bag of Words Representation ---")
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(demo_documents)

print("Vocabulary Found :", bow_vectorizer.get_feature_names_out())
with np.printoptions(precision=3, suppress=True, floatmode="fixed"):
    print("BoW Matrix Array :\n", bow_matrix.toarray().astype(float))

print("\n--- 2. TF-IDF Representation ---")
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(demo_documents)

with np.printoptions(precision=3, suppress=True, floatmode="fixed"):
    print("TF-IDF Matrix Array:\n", tfidf_matrix.toarray())

--- 1. Bag of Words Representation ---
Vocabulary Found : ['acknowledged' 'anonymous' 'answer' 'austen' 'bennet' 'chapter'
 'considered' 'daughter' 'day' 'dear' 'entering' 'family' 'feeling'
 'first' 'fixed' 'fortune' 'good' 'heard' 'however' 'jane' 'known' 'lady'
 'last' 'let' 'little' 'long' 'made' 'man' 'may' 'mind' 'mr' 'must'
 'neighbourhood' 'netherfield' 'one' 'park' 'possession' 'prejudice'
 'pride' 'produced' 'property' 'replied' 'returned' 'rightful' 'said'
 'single' 'surrounding' 'told' 'truth' 'universally' 'view' 'volunteer'
 'want' 'well' 'wife']
BoW Matrix Array :
 [[1.000 1.000 0.000 1.000 0.000 1.000 0.000 0.000 0.000 0.000 0.000 0.000
  0.000 0.000 0.000 1.000 1.000 0.000 0.000 1.000 0.000 0.000 0.000 0.000
  0.000 0.000 0.000 1.000 0.000 0.000 0.000 1.000 0.000 0.000 0.000 0.000
  1.000 1.000 1.000 1.000 0.000 0.000 0.000 0.000 0.000 1.000 0.000 0.000
  1.000 1.000 0.000 1.000 1.000 0.000 1.000]
 [0.000 0.000 0.000 0.000 0.000 0.000 1.000 1.000 0.000 0.000 1.000 1.00

## 5. Levenshtein Distance (Corpus-Only Correction)

`calculate_edit_distance` computes the minimum-edit-distance between two strings via
dynamic programming; `correct_word_edit_distance` uses it to find the closest word
that already exists in `base_vocabulary`.


In [6]:
def calculate_edit_distance(word1, word2):

    m, n = len(word1), len(word2)

    dp = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[m][n]


def correct_word_edit_distance(word, vocab):
    """Corpus-only spelling correction (original approach): finds the
    closest word that already exists in the training vocabulary."""

    if word in vocab:
        return word

    distances = {}

    for valid_word in vocab:

        if valid_word == "<s>" or valid_word == "</s>":
            continue

        distance = calculate_edit_distance(word, valid_word)
        distances[valid_word] = distance

    best_word = min(distances, key=distances.get)

    return best_word


# --- Sample test cases (misspellings of words that ARE in this corpus) ---
edit_distance_test_words = ["elisabeth", "darcey", "familly", "sencible"]

print("Edit distance:", calculate_edit_distance("kitten", "sitting"), "(kitten -> sitting)")
print()
print("Corpus-based corrections:")
for test_word in edit_distance_test_words:
    corrected = correct_word_edit_distance(test_word, base_vocabulary)
    print(f"  '{test_word}' -> '{corrected}'")

Edit distance: 3 (kitten -> sitting)

Corpus-based corrections:
  'elisabeth' -> 'elizabeth'
  'darcey' -> 'darcy'
  'familly' -> 'family'
  'sencible' -> 'sensible'


In [29]:
def calculate_edit_distance(word1, word2, show_table=True):
    """
    Calculate Levenshtein edit distance between two words.
    Optionally display the dynamic programming table.
    """
    word1 = word1.lower()
    word2 = word2.lower()

    m, n = len(word1), len(word2)

    dp = np.zeros((m + 1, n + 1), dtype=int)

    # Initialization
    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    # Fill DP table
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    # Display DP table
    if show_table:
        rows = ["∅"] + list(word1)
        cols = ["∅"] + list(word2)

        dp_table = pd.DataFrame(
            dp,
            index=rows,
            columns=cols
        )

        print(f"\nEdit Distance Table: '{word1}' → '{word2}'")
        display(dp_table)

    return dp[m][n]


def correct_word_edit_distance(word, vocab):
    """
    Find the closest word from the vocabulary
    using Levenshtein edit distance.
    """
    word = word.lower()

    if word in vocab:
        return word, 0

    distances = {}

    for valid_word in vocab:

        # Ignore sentence boundary tokens
        if valid_word in {"<s>", "</s>"}:
            continue

        distance = calculate_edit_distance(
            word,
            valid_word,
            show_table=False
        )

        distances[valid_word] = distance

    best_word = min(distances, key=distances.get)

    return best_word, distances[best_word]

In [31]:
# --- Sample test cases (misspellings of words that ARE in this corpus) ---
edit_distance_test_words = ["elisabeth", "darcey", "familly", "sencible"]

print("Edit distance:", calculate_edit_distance("kitten", "sitting", show_table=True), "(kitten -> sitting)")
print()
print("Corpus-based corrections:")
for test_word in edit_distance_test_words:
    corrected_word, distance = correct_word_edit_distance(test_word, base_vocabulary)
    print(f"  '{test_word}' -> '{corrected_word}' (distance: {distance})")


Edit Distance Table: 'kitten' → 'sitting'


,∅,s,i,t,t,i,n,g
∅,0,1,2,3,4,5,6,7
k,1,1,2,3,4,5,6,7
i,2,2,1,2,3,4,5,6
t,3,3,2,1,2,3,4,5
t,4,4,3,2,1,2,3,4
e,5,5,4,3,2,2,3,4
n,6,6,5,4,3,3,2,3


Edit distance: 3 (kitten -> sitting)

Corpus-based corrections:
  'elisabeth' -> 'elizabeth' (distance: 1)
  'darcey' -> 'darcy' (distance: 1)
  'familly' -> 'family' (distance: 1)
  'sencible' -> 'sensible' (distance: 1)


In [33]:
print("=== SPELLING CORRECTION ===")

try:
    user_word = input("Enter a word to check: ").strip().lower()
except EOFError:
    user_word = "sencible"
    print("No input provided; using sample word: sencible")

corrected_word, distance = correct_word_edit_distance(
    user_word,
    base_vocabulary
)

calculate_edit_distance(user_word, corrected_word, show_table=True)

if user_word == corrected_word:
    print(f"\n'{user_word}' is already correct.")
else:
    print(f"\nInput word       : {user_word}")
    print(f"Corrected word   : {corrected_word}")
    print(f"Edit distance    : {distance}")

=== SPELLING CORRECTION ===

Edit Distance Table: 'elisabeth' → 'elizabeth'


,∅,e,l,i,z,a,b,e,t,h
∅,0,1,2,3,4,5,6,7,8,9
e,1,0,1,2,3,4,5,6,7,8
l,2,1,0,1,2,3,4,5,6,7
i,3,2,1,0,1,2,3,4,5,6
s,4,3,2,1,1,2,3,4,5,6
a,5,4,3,2,2,1,2,3,4,5
b,6,5,4,3,3,2,1,2,3,4
e,7,6,5,4,4,3,2,1,2,3
t,8,7,6,5,5,4,3,2,1,2
h,9,8,7,6,6,5,4,3,2,1



Input word       : elisabeth
Corrected word   : elizabeth
Edit distance    : 1


In [34]:
print("\n=== EDIT DISTANCE CALCULATOR ===")

try:
    word1 = input("Enter first word: ").strip()
    word2 = input("Enter second word: ").strip()
except EOFError:
    word1, word2 = "kitten", "sitting"
    print("No input provided; using sample words: kitten and sitting")

distance = calculate_edit_distance(
    word1,
    word2,
    show_table=True
)

print(f"\nEdit distance between '{word1}' and '{word2}': {distance}")


=== EDIT DISTANCE CALCULATOR ===

Edit Distance Table: 'kitten' → 'sitting'


,∅,s,i,t,t,i,n,g
∅,0,1,2,3,4,5,6,7
k,1,1,2,3,4,5,6,7
i,2,2,1,2,3,4,5,6
t,3,3,2,1,2,3,4,5
t,4,4,3,2,1,2,3,4
e,5,5,4,3,2,2,3,4
n,6,6,5,4,3,3,2,3



Edit distance between 'kitten' and 'sitting': 3


## 6. Dictionary-Based Spell Checking

The corpus-only corrector above can only fix a typo if the *correct* spelling already
happens to appear in `corpus.txt`. That fails for a perfectly ordinary English word the
corpus never used (e.g. `langauge -> language`, `machne -> machine` — neither word
appears in *Pride and Prejudice*).

[`pyspellchecker`](https://pypi.org/project/pyspellchecker/) ships with a large,
general-English word-frequency dictionary, so it can correct such typos even when the
correct word never appears in the corpus.

**Correction strategy** (best of both worlds):
1. If the word is already in the corpus vocabulary, keep it unchanged.
2. Otherwise, ask `pyspellchecker` for its best general-English dictionary correction.
3. If the dictionary has no suggestion at all, fall back to the corpus edit-distance
   corrector from Section 5 so the word still maps to *some* known token.


In [30]:
import sys
import subprocess

try:
    from spellchecker import SpellChecker
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspellchecker", "-q"])
    from spellchecker import SpellChecker

spell = SpellChecker()


def correct_word(word, vocab, spell_checker=spell):
    """Correct `word` using the corpus vocabulary first, then a general
    English dictionary, falling back to corpus edit-distance if the
    dictionary has no suggestion."""

    word = word.lower()

    if word in vocab:
        return word

    dictionary_correction = spell_checker.correction(word)

    if dictionary_correction is None:
        corrected = correct_word_edit_distance(word, vocab)
        print(f"Corrected '{word}' to '{corrected}' (corpus edit-distance)")
        return corrected

    if dictionary_correction != word:
        print(f"Corrected '{word}' to '{dictionary_correction}' (dictionary)")

    return dictionary_correction

In [32]:
# --- Sample test cases (words absent from the corpus but valid English) ---
dictionary_test_words = ["langauge", "machne", "recieve", "definately"]

print("Dictionary-based corrections:")
for test_word in dictionary_test_words:
    corrected_word = correct_word(test_word, base_vocabulary)
    print(f"  '{test_word}' -> '{corrected_word}'")

Dictionary-based corrections:
Corrected 'langauge' to 'language' (dictionary)
  'langauge' -> 'language'
Corrected 'machne' to 'machine' (dictionary)
  'machne' -> 'machine'
Corrected 'recieve' to 'receive' (dictionary)
  'recieve' -> 'receive'
Corrected 'definately' to 'definitely' (dictionary)
  'definately' -> 'definitely'


In [35]:
print("\n=== DICTIONARY-BASED SPELL CHECKING ===")

try:
    user_word = input("Enter a word to check: ").strip().lower()
except EOFError:
    user_word = "langauge"
    print("No input provided; using sample word: langauge")

corrected_word = correct_word(user_word, base_vocabulary)
distance = calculate_edit_distance(user_word, corrected_word, show_table=True)

if user_word == corrected_word:
    print(f"\n'{user_word}' is already correct.")
else:
    print(f"\nInput word       : {user_word}")
    print(f"Corrected word   : {corrected_word}")
    print(f"Edit distance    : {distance}")


=== DICTIONARY-BASED SPELL CHECKING ===

Edit Distance Table: 'produced' → 'produced'


,∅,p,r,o,d,u,c,e,d
∅,0,1,2,3,4,5,6,7,8
p,1,0,1,2,3,4,5,6,7
r,2,1,0,1,2,3,4,5,6
o,3,2,1,0,1,2,3,4,5
d,4,3,2,1,0,1,2,3,4
u,5,4,3,2,1,0,1,2,3
c,6,5,4,3,2,1,0,1,2
e,7,6,5,4,3,2,1,0,1
d,8,7,6,5,4,3,2,1,0



'produced' is already correct.


## 7. Automatic N Detection + User Input

The user simply types their sentence so far. The n-gram order `N` is set to the
number of words they typed (Requirement 1) — there is **no** prompt asking for `N`,
and **no** artificial upper bound on how large `N` can be; it is determined purely by
how many words the user types. Each word is spell-corrected with the dictionary-aware
corrector from Section 6.

| Words typed | N (order) used |
|---|---|
| `I` | 1 (unigram) |
| `I love` | 2 (bigram) |
| `I love natural` | 3 (trigram) |
| `I love natural language` | 4 (4-gram) |


In [8]:
raw_input_text = input("Enter your sentence so far: ").strip()

while not raw_input_text:
    print("Please type at least one word.")
    raw_input_text = input("Enter your sentence so far: ").strip()

typed_words = raw_input_text.lower().split()

N = len(typed_words)  # <-- automatically detected, never asked for, never capped
print(f"Detected n-gram order: N = {N}")

corrected_words = [correct_word(word, base_vocabulary) for word in typed_words]

print("Corrected words:", corrected_words)

Detected n-gram order: N = 3
Corrected words: ['but', 'it', 'is']


## 8. Sentence Splitting & Preprocessing for the Language Model

Now that `N` is known (Section 7), every corpus sentence is padded with **exactly**
`N - 1` `<s>` markers — no more, no less — so the printed tokens stay readable no
matter what `N` turns out to be.


In [9]:
def preprocess_text(text, n):

    sentences = sent_tokenize(text)

    processed_sentences = []

    for sentence in sentences:

        sentence = clean_sentence(sentence)

        start_markers = " ".join(["<s>"] * (n - 1))
        sentence = (start_markers + " " + sentence + " </s>").strip()

        tokens = sentence.split()

        if tokens:
            processed_sentences.append(tokens)

    return processed_sentences


sentences = preprocess_text(raw_text, N)

print(f"Padded each sentence with {N - 1} '<s>' marker(s), based on the detected N = {N}.")
print(sentences[:3])

Padded each sentence with 2 '<s>' marker(s), based on the detected N = 3.
[['<s>', '<s>', 'produced', 'by', 'anonymous', 'volunteers', 'pride', 'and', 'prejudice', 'by', 'jane', 'austen', 'chapter', 'it', 'is', 'a', 'truth', 'universally', 'acknowledged', 'that', 'a', 'single', 'man', 'in', 'possession', 'of', 'a', 'good', 'fortune', 'must', 'be', 'in', 'want', 'of', 'a', 'wife', '</s>'], ['<s>', '<s>', 'however', 'little', 'known', 'the', 'feelings', 'or', 'views', 'of', 'such', 'a', 'man', 'may', 'be', 'on', 'his', 'first', 'entering', 'a', 'neighbourhood', 'this', 'truth', 'is', 'so', 'well', 'fixed', 'in', 'the', 'minds', 'of', 'the', 'surrounding', 'families', 'that', 'he', 'is', 'considered', 'the', 'rightful', 'property', 'of', 'some', 'one', 'or', 'other', 'of', 'their', 'daughters', '</s>'], ['<s>', '<s>', 'my', 'dear', 'mr', 'bennet', 'said', 'his', 'lady', 'to', 'him', 'one', 'day', 'have', 'you', 'heard', 'that', 'netherfield', 'park', 'is', 'let', 'at', 'last', '</s>']]


In [10]:
tokens = []

for sentence in sentences:
    tokens.extend(sentence)

ngram_vocabulary = set(tokens)

# Vocabulary used for NEXT-WORD prediction/sampling should never include the
# sentence-start marker -- a real generated word should never be "<s>".
predictable_vocab = sorted(ngram_vocabulary - {"<s>"})

print("Total tokens:", len(tokens))
print("N-gram vocabulary size:", len(ngram_vocabulary))
print("Predictable vocabulary size:", len(predictable_vocab))

Total tokens: 139405
N-gram vocabulary size: 6820
Predictable vocabulary size: 6819


## 9. Dynamic N-Gram Counting (Any Order)

Rather than writing separate code for unigram / bigram / trigram counting (or
hardcoding a single fixed order), one generalized function builds the count table for
*any* history length. It is reused for every order from `0` (unigram, empty history)
up to `N - 1` — exactly the orders needed for backoff on this specific query, nothing
more.


In [11]:
def build_ngram_counts(sentence_list, history_length):
    """Generalized n-gram counter.

    history_length = 0 -> unigram counts (history is the empty tuple)
    history_length = 1 -> bigram counts
    history_length = k -> (k+1)-gram counts
    """
    count_table = defaultdict(Counter)

    for sentence in sentence_list:
        for i in range(history_length, len(sentence)):

            history = tuple(sentence[i - history_length:i])
            next_word = sentence[i]

            if next_word == "<s>":
                continue

            count_table[history][next_word] += 1

    return count_table


count_tables = {
    history_length: build_ngram_counts(sentences, history_length)
    for history_length in range(N)
}

print(f"Built dynamic n-gram count tables for orders 1 through {N} (history lengths 0-{N - 1}).")

# --- Sample test case: raw counts for a couple of concrete histories ---
print("\nSample counts, history=() [[unigram]]:", count_tables[0][()].most_common(5))

if N >= 2:
    example_bigram_history = ("the",)
    print(f"Sample counts, history={example_bigram_history} [[bigram]]:",
          count_tables[1][example_bigram_history].most_common(5))

Built dynamic n-gram count tables for orders 1 through 3 (history lengths 0-2).

Sample counts, history=() [[unigram]]: [('</s>', 5976), ('the', 4321), ('to', 4127), ('of', 3597), ('and', 3531)]
Sample counts, history=('the',) [[bigram]]: [('whole', 72), ('room', 69), ('same', 69), ('world', 63), ('first', 57)]


## 10. N-Gram Probability Tables (Unsmoothed MLE)

For a history `h`, the Maximum Likelihood Estimate of the next word `w` is simply

$$P_{MLE}(w \mid h) = \frac{C(h, w)}{C(h)}$$

Below, a probability table is printed **for every order from unigram up to the
detected `N`-gram** — i.e. exactly the orders that exist for this query (type a
longer sentence in Section 7 to see more tables: 2 words -> unigram + bigram,
3 words -> unigram + bigram + trigram, and so on).

For each order, the history with the most observed continuations is picked so the
table has several rows worth looking at.


In [12]:
def build_probability_table(count_table):
    """Convert a raw count table into an MLE probability table."""
    probability_table = defaultdict(dict)
    for history, next_word_counts in count_table.items():
        total = sum(next_word_counts.values())
        for word, count in next_word_counts.items():
            probability_table[history][word] = count / total
    return probability_table


probability_tables = {
    history_length: build_probability_table(count_tables[history_length])
    for history_length in range(N)
}

order_names = {0: "Unigram", 1: "Bigram", 2: "Trigram"}


def order_label(history_length):
    return order_names.get(history_length, f"{history_length + 1}-gram")


example_histories = {}

for history_length in range(N):
    table = count_tables[history_length]
    if not table:
        continue

    # Pick the history with the richest set of observed continuations
    example_history = max(table, key=lambda h: sum(table[h].values()))
    example_histories[history_length] = example_history

    print(f"--- {order_label(history_length)} Probability Table (unsmoothed MLE) ---")
    print(f"History: {example_history}")
    for word, prob in sorted(
        probability_tables[history_length][example_history].items(),
        key=lambda item: -item[1]
    )[:8]:
        print(f"  P({word} | {example_history}) = {prob:.4f}")
    print()

--- Unigram Probability Table (unsmoothed MLE) ---
History: ()
  P(</s> | ()) = 0.0469
  P(the | ()) = 0.0339
  P(to | ()) = 0.0324
  P(of | ()) = 0.0282
  P(and | ()) = 0.0277
  P(her | ()) = 0.0174
  P(i | ()) = 0.0161
  P(a | ()) = 0.0152

--- Bigram Probability Table (unsmoothed MLE) ---
History: ('<s>',)
  P(i | ('<s>',)) = 0.1017
  P(she | ('<s>',)) = 0.0549
  P(but | ('<s>',)) = 0.0529
  P(the | ('<s>',)) = 0.0440
  P(it | ('<s>',)) = 0.0408
  P(he | ('<s>',)) = 0.0393
  P(you | ('<s>',)) = 0.0333
  P(elizabeth | ('<s>',)) = 0.0315

--- Trigram Probability Table (unsmoothed MLE) ---
History: ('<s>', '<s>')
  P(i | ('<s>', '<s>')) = 0.1017
  P(she | ('<s>', '<s>')) = 0.0549
  P(but | ('<s>', '<s>')) = 0.0529
  P(the | ('<s>', '<s>')) = 0.0440
  P(it | ('<s>', '<s>')) = 0.0408
  P(he | ('<s>', '<s>')) = 0.0393
  P(you | ('<s>', '<s>')) = 0.0333
  P(elizabeth | ('<s>', '<s>')) = 0.0315



## 11. Laplace (Add-One) Smoothing

The unsmoothed tables above assign **zero** probability to any word that was never
observed after a given history `h`. Laplace smoothing fixes this by adding 1 to every
count and adding the vocabulary size `V` to the denominator, so every word in the
vocabulary keeps a small, non-zero probability:

$$P_{Laplace}(w \mid h) = \frac{C(h, w) + 1}{C(h) + V}$$

This generalizes to *any* n-gram order because `count_tables[history_length]` already
stores `C(h, w)` for that specific order — only `V` (size of the predictable
vocabulary) is shared across every order. Below, the **same example histories** from
Section 10 are shown again with Laplace smoothing applied, so you can directly compare
the unsmoothed and smoothed probability tables.


In [13]:
VOCAB_SIZE = len(predictable_vocab)


def laplace_probability(word, history, count_table, vocab_size=VOCAB_SIZE, k=1):
    """Add-k (Laplace when k=1) smoothed probability of `word` following `history`."""
    counts = count_table.get(history, Counter())
    total = sum(counts.values())
    return (counts[word] + k) / (total + k * vocab_size)


def get_smoothed_distribution(history, count_table, vocab, k=1):
    """Full Laplace-smoothed probability distribution over `vocab` for a history."""
    counts = count_table.get(history, Counter())
    total = sum(counts.values())
    v = len(vocab)
    return {word: (counts[word] + k) / (total + k * v) for word in vocab}


for history_length, example_history in example_histories.items():
    smoothed_distribution = get_smoothed_distribution(
        example_history, count_tables[history_length], predictable_vocab
    )
    print(f"--- {order_label(history_length)} Probability Table (Laplace-smoothed) ---")
    print(f"History: {example_history}")
    for word, prob in Counter(smoothed_distribution).most_common(8):
        print(f"  P({word} | {example_history}) = {prob:.5f}")
    print()

--- Unigram Probability Table (Laplace-smoothed) ---
History: ()
  P(</s> | ()) = 0.04451
  P(the | ()) = 0.03219
  P(to | ()) = 0.03074
  P(of | ()) = 0.02680
  P(and | ()) = 0.02630
  P(her | ()) = 0.01650
  P(i | ()) = 0.01527
  P(a | ()) = 0.01445

--- Bigram Probability Table (Laplace-smoothed) ---
History: ('<s>',)
  P(i | ('<s>',)) = 0.04760
  P(she | ('<s>',)) = 0.02571
  P(but | ('<s>',)) = 0.02478
  P(the | ('<s>',)) = 0.02063
  P(it | ('<s>',)) = 0.01915
  P(he | ('<s>',)) = 0.01844
  P(you | ('<s>',)) = 0.01563
  P(elizabeth | ('<s>',)) = 0.01477

--- Trigram Probability Table (Laplace-smoothed) ---
History: ('<s>', '<s>')
  P(i | ('<s>', '<s>')) = 0.04760
  P(she | ('<s>', '<s>')) = 0.02571
  P(but | ('<s>', '<s>')) = 0.02478
  P(the | ('<s>', '<s>')) = 0.02063
  P(it | ('<s>', '<s>')) = 0.01915
  P(he | ('<s>', '<s>')) = 0.01844
  P(you | ('<s>', '<s>')) = 0.01563
  P(elizabeth | ('<s>', '<s>')) = 0.01477



## 12. Non-Deterministic Next-Word Prediction (Shannon's-Game Style)

Always picking `argmax P(w | h)` makes the same input produce the same output every
time. Instead, the teacher wants **weighted random sampling**: words with higher
probability should be picked more often, but lower-probability words should still have
a chance of appearing.

`random.choices(population, weights=...)` implements exactly that. `sample_next_word`
takes an `rng` object (defaulting to the plain `random` module) so a seed is entirely
**optional** — pass a seeded `random.Random(seed)` for reproducible output, or leave
the default for a different result on every run.

`shannons_predict` wraps this into a single-step "next word" demonstration, similar to
Claude Shannon's classic guessing game: it shows the top candidate words with their
smoothed probabilities, then samples one.


In [14]:
def sample_next_word(distribution, rng=random):
    """Weighted random sample of the next word from a probability distribution."""
    words = list(distribution.keys())
    weights = list(distribution.values())
    return rng.choices(words, weights=weights, k=1)[0]


def shannons_predict(history_words, count_tables, vocab, k=1, top_n=5):
    """Single next-word prediction step: find the best available (backoff)
    history, show its top smoothed candidates, then sample one word."""

    max_history_length = min(len(history_words), N - 1)

    chosen_history_length, chosen_history = None, None

    for history_length in range(max_history_length, -1, -1):
        history = tuple(history_words[-history_length:]) if history_length else ()

        if history in count_tables[history_length] and count_tables[history_length][history]:
            chosen_history_length, chosen_history = history_length, history
            break

    if chosen_history_length is None:
        print("No continuation was found for this history.")
        return None

    distribution = get_smoothed_distribution(
        chosen_history, count_tables[chosen_history_length], vocab, k
    )

    print(f"History used: {chosen_history} (order {chosen_history_length + 1} model)")
    print("Top candidates:")
    for word, prob in Counter(distribution).most_common(top_n):
        print(f"  {word:<12} {prob:.4f}")

    predicted_word = sample_next_word(distribution)
    print(f"Sampled next word: '{predicted_word}'")

    return predicted_word


# --- Sample test case: run this cell a few times, the sampled word can change ---
_ = shannons_predict(corrected_words, count_tables, predictable_vocab)

History used: ('it', 'is') (order 3 model)
Top candidates:
  a            0.0030
  not          0.0029
  very         0.0021
  the          0.0011
  impossible   0.0009
Sampled next word: 'neighbouring'


## 13. Sentence Generation (Backoff + Smoothing + Sampling)

Starting from the corrected input words (Section 7), the notebook repeatedly:

1. Tries the highest order first: the last `N-1` words as history.
2. If that exact history was never observed, backs off to a shorter history
   (`N-2`, `N-3`, ... down to the empty/unigram history, which always exists).
3. Builds the Laplace-smoothed distribution over the whole predictable vocabulary for
   the chosen history (Section 11).
4. Samples the next word from that distribution (Section 12) instead of always taking
   the single most probable word.

Generation stops at `</s>` or after `MAX_GENERATED_WORDS` words.


In [15]:
MAX_GENERATED_WORDS = 50
GENERATION_SEED = None  # set an int for reproducible generation, or leave None

# A single rng is created for the whole generation so an optional seed
# reproduces the entire sentence, not just a single word.
rng = random.Random(GENERATION_SEED) if GENERATION_SEED is not None else random

# Full context is kept for display; the loop below only ever looks back at the
# last (N-1) tokens of it when building n-gram history, exactly matching an
# order-N model.
generated_words = list(corrected_words)

reached_end_marker = False

for _ in range(MAX_GENERATED_WORDS):

    chosen_history_length = None
    chosen_history = None

    for history_length in range(N - 1, -1, -1):

        history = tuple(generated_words[-history_length:]) if history_length else ()

        if history in count_tables[history_length] and count_tables[history_length][history]:
            chosen_history_length = history_length
            chosen_history = history
            break

    if chosen_history_length is None:
        print("No continuation was found; stopping generation.")
        break

    distribution = get_smoothed_distribution(
        chosen_history, count_tables[chosen_history_length], predictable_vocab
    )

    next_word = sample_next_word(distribution, rng=rng)

    generated_words.append(next_word)

    if next_word == "</s>":
        reached_end_marker = True
        break

if not reached_end_marker:
    print(f"Stopped after {MAX_GENERATED_WORDS} words to avoid an infinite loop.")


output_words = [word for word in generated_words if word not in ("<s>", "</s>")]
generated_sentence = " ".join(output_words)

print("\nGenerated Sentence:")
print(generated_sentence)

Stopped after 50 words to avoid an infinite loop.

Generated Sentence:
but it is plenty laudable childhood longing lodgings using dearly yesthe relied farmhouse perverse meditate arrear shortness perfections boy thus gallant rendered friendalways sloping digressions shelves chiefly inherit master's remaining mixing accede vogue soon build accents thanking horse heartfelt thursday deaden pocket dawdled gratitudefor understand rapidly expressions overruled guess curtsey sentinel cheapside stop
